# **1. Perkenalan Dataset**

Dataset yang digunakan pada eksperimen ini adalah **Titanic Dataset**, salah satu dataset paling populer di dunia machine learning. Dataset ini berisi informasi penumpang kapal Titanic dan digunakan untuk memprediksi apakah seorang penumpang selamat atau tidak dari tragedi tenggelamnya kapal Titanic pada tahun 1912.

**Sumber Dataset:** [Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic) dan [GitHub datasciencedojo](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv)

**Jumlah Data:** 891 baris, 12 kolom

**Deskripsi Fitur:**
| Kolom | Deskripsi |
|---|---|
| PassengerId | ID unik penumpang |
| Survived | Target (0 = Meninggal, 1 = Selamat) |
| Pclass | Kelas tiket (1 = 1st, 2 = 2nd, 3 = 3rd) |
| Name | Nama penumpang |
| Sex | Jenis kelamin |
| Age | Usia penumpang |
| SibSp | Jumlah saudara/pasangan di atas kapal |
| Parch | Jumlah orang tua/anak di atas kapal |
| Ticket | Nomor tiket |
| Fare | Harga tiket |
| Cabin | Nomor kabin |
| Embarked | Pelabuhan keberangkatan (C, Q, S) |

# **2. Import Library**

Pada tahap ini, kita mengimpor semua pustaka Python yang diperlukan untuk analisis data, visualisasi, dan preprocessing.

In [ ]:
# Import library dasar untuk manipulasi data
import pandas as pd
import numpy as np

# Import library untuk visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns

# Import library untuk Data Preprocessing dari Scikit-Learn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Pengaturan untuk mempercantik visualisasi dan mengabaikan peringatan
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimpor!")

# **3. Memuat Dataset**

Dataset Titanic dimuat langsung dari URL publik menggunakan `pandas.read_csv()`. Setelah dimuat, kita periksa beberapa baris awal dan dimensi dataset untuk memastikan data berhasil dimuat dengan benar.

In [ ]:
# Memuat dataset Titanic dari URL publik
dataset_path = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(dataset_path)

# Menampilkan 5 baris pertama dataset
print("Tampilan 5 Baris Pertama Dataset:")
display(df.head())

# Memeriksa dimensi dataset
print(f"\nDimensi dataset: {df.shape[0]} baris dan {df.shape[1]} kolom.")

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset secara mendalam sebelum melakukan preprocessing. EDA membantu kita:
- Memahami struktur dan tipe data
- Menemukan missing values dan duplikat
- Melihat distribusi fitur
- Memahami hubungan antar variabel

In [ ]:
# === 4.1 Informasi Umum Dataset ===
print("=" * 50)
print("INFORMASI DATASET")
print("=" * 50)
df.info()

print("\n=" * 50)
print("STATISTIK DESKRIPTIF (KOLOM NUMERIK)")
print("=" * 50)
display(df.describe())

In [ ]:
# === 4.2 Analisis Missing Values ===
print("=" * 50)
print("ANALISIS MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Jumlah Missing', ascending=False)
print(missing_df)

print(f"\nTotal duplikat: {df.duplicated().sum()}")

# Visualisasi Missing Values
plt.figure(figsize=(10, 4))
missing_df['Persentase (%)'].plot(kind='bar', color=['#e74c3c', '#e67e22', '#f1c40f'])
plt.title('Persentase Missing Values per Kolom')
plt.xlabel('Kolom')
plt.ylabel('Persentase (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === 4.3 Distribusi Fitur Numerik ===
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold')

num_features = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass', 'PassengerId']
for ax, col in zip(axes.flatten(), num_features):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribusi {col}')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# === 4.4 Distribusi Fitur Kategorikal & Target ===
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Distribusi Fitur Kategorikal & Target', fontsize=14, fontweight='bold')

# Distribusi Target (Survived)
survived_counts = df['Survived'].value_counts()
axes[0].pie(survived_counts, labels=['Meninggal (0)', 'Selamat (1)'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
axes[0].set_title('Distribusi Target (Survived)')

# Distribusi Jenis Kelamin
sns.countplot(data=df, x='Sex', hue='Survived', ax=axes[1],
              palette={0: '#e74c3c', 1: '#2ecc71'})
axes[1].set_title('Survived berdasarkan Jenis Kelamin')
axes[1].legend(['Meninggal', 'Selamat'])

# Distribusi Kelas Tiket
sns.countplot(data=df, x='Pclass', hue='Survived', ax=axes[2],
              palette={0: '#e74c3c', 1: '#2ecc71'})
axes[2].set_title('Survived berdasarkan Kelas Tiket')
axes[2].legend(['Meninggal', 'Selamat'])

plt.tight_layout()
plt.show()

In [ ]:
# === 4.5 Matriks Korelasi ===
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Matriks Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi terhadap target
print("\nKorelasi Fitur terhadap Target (Survived):")
print(corr_matrix['Survived'].sort_values(ascending=False))

In [ ]:
# === 4.6 Deteksi Outlier dengan Boxplot ===
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Deteksi Outlier — Boxplot', fontsize=14, fontweight='bold')

outlier_cols = ['Age', 'Fare', 'SibSp']
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col].dropna(), ax=ax, color='lightblue')
    ax.set_title(f'Boxplot {col}')

plt.tight_layout()
plt.show()

# Ringkasan temuan EDA
print("\n=== RINGKASAN TEMUAN EDA ===")
print(f"1. Dataset memiliki {df.shape[0]} baris dan {df.shape[1]} kolom")
print(f"2. Kolom dengan missing values: Age ({df['Age'].isnull().sum()}), Cabin ({df['Cabin'].isnull().sum()}), Embarked ({df['Embarked'].isnull().sum()})")
print(f"3. Tidak ada duplikat: {df.duplicated().sum()} duplikat")
print(f"4. Rasio kelas target: {(df['Survived']==1).sum()} selamat ({(df['Survived']==1).mean()*100:.1f}%), {(df['Survived']==0).sum()} meninggal")
print(f"5. Kolom 'Fare' dan 'Age' memiliki outlier signifikan")
print(f"6. Kolom kategorikal yang perlu encoding: Sex, Cabin, Embarked, Name, Ticket")

# **5. Data Preprocessing**

Berdasarkan temuan EDA, kita melakukan preprocessing dengan tahapan berikut:

1. **Menghapus Duplikat** — Tidak ditemukan duplikat, tapi tetap dijalankan sebagai langkah standar
2. **Menangani Missing Values** — Imputasi median untuk numerik, modus untuk kategorikal
3. **Encoding Kategorikal** — Label Encoding untuk kolom teks
4. **Standarisasi Fitur** — StandardScaler agar skala data seragam (Mean=0, Std=1)
5. **Menyimpan Hasil** — Output dataset siap latih ke file CSV

In [ ]:
# Membuat salinan dataset agar data asli tetap utuh
df_clean = df.copy()

# === STEP 1: Menghapus Data Duplikat ===
jumlah_duplikat = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"Step 1 — Duplikat dihapus: {jumlah_duplikat} baris")
print(f"         Ukuran data setelah: {df_clean.shape}")

In [ ]:
# === STEP 2: Menangani Missing Values ===
print("Missing Values SEBELUM imputasi:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Pisahkan kolom berdasarkan tipe data
target_col = 'Survived'
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=[np.number]).columns.tolist()

# Keluarkan target dari proses preprocessing fitur
if target_col in num_cols:
    num_cols.remove(target_col)

# Imputasi kolom numerik dengan Median
imputer_num = SimpleImputer(strategy='median')
if len(num_cols) > 0:
    df_clean[num_cols] = imputer_num.fit_transform(df_clean[num_cols])

# Imputasi kolom kategorikal dengan Modus
imputer_cat = SimpleImputer(strategy='most_frequent')
if len(cat_cols) > 0:
    df_clean[cat_cols] = imputer_cat.fit_transform(df_clean[cat_cols])

print("\nMissing Values SESUDAH imputasi:")
print(df_clean.isnull().sum().sum(), "— tidak ada missing value tersisa")

In [ ]:
# === STEP 3: Encoding Data Kategorikal ===
print("Kolom kategorikal yang akan di-encode:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f"   Encoded: {col}")

print("\nTipe data setelah encoding:")
print(df_clean.dtypes)

In [ ]:
# === STEP 4: Standarisasi Fitur (StandardScaler) ===
# Keluarkan kolom target dari proses scaling
features_to_scale = [col for col in df_clean.columns if col != target_col]

scaler = StandardScaler()
df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])

print("Standarisasi selesai. Statistik fitur setelah scaling:")
display(df_clean[features_to_scale].describe().round(4))

In [ ]:
# === STEP 5: Menyimpan Hasil Preprocessing ===
output_path = "titanic_preprocessing.csv"
df_clean.to_csv(output_path, index=False)

print("=" * 50)
print("PREPROCESSING SELESAI!")
print("=" * 50)
print(f"Dataset hasil preprocessing disimpan di: {output_path}")
print(f"Dimensi akhir: {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom")
print(f"\nTampilan 5 baris pertama data siap latih:")
display(df_clean.head())